# London Rail Network - Train Animations
## Animated Train Movements on Satellite Imagery

This notebook shows trains moving along the network based on timetable data, overlaid on satellite imagery.

**Days covered:** Monday, Friday, Saturday, Sunday

In [ ]:
# Install dependencies if needed
try:
    import contextily as ctx
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'contextily'])
    import contextily as ctx

import pyarrow.feather as feather
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.lines import Line2D
from pathlib import Path
from datetime import datetime, timedelta
from IPython.display import HTML
import re

# Coordinate transformations
from pyproj import Transformer

# Set up paths
DATA_DIR = Path('../data/processed_data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Transformers
osgb_to_webmerc = Transformer.from_crs('EPSG:27700', 'EPSG:3857', always_xy=True)
wgs84_to_webmerc = Transformer.from_crs('EPSG:4326', 'EPSG:3857', always_xy=True)

print("Dependencies loaded")

In [ ]:
# === LINE DEFINITIONS ===

ALL_LINES = {
    # Underground
    'BAK': ('#B36305', 'Bakerloo'),
    'CEN': ('#E32017', 'Central'),
    'CIR': ('#FFD300', 'Circle'),
    'DIS': ('#00782A', 'District'),
    'HAM': ('#F3A9BB', 'Hammersmith & City'),
    'JUB': ('#A0A5A9', 'Jubilee'),
    'MET': ('#9B0056', 'Metropolitan'),
    'NTN': ('#000000', 'Northern'),
    'PIC': ('#003688', 'Piccadilly'),
    'VIC': ('#0098D4', 'Victoria'),
    'WAC': ('#95CDBA', 'Waterloo & City'),
    # Elizabeth Line
    'ELZ': ('#6950A1', 'Elizabeth'),
    # DLR
    'DLR': ('#00A4A7', 'DLR'),
    # Trams
    'TRM': ('#84B817', 'Trams'),
    # Overground
    'LIB': ('#6B717A', 'Liberty'),
    'LIO': ('#FFA600', 'Lioness'),
    'MIL': ('#00A0E2', 'Mildmay'),
    'SUF': ('#00853E', 'Suffragette'),
    'WEA': ('#9B0058', 'Weaver'),
    'WIN': ('#E21836', 'Windrush'),
}

## Load Station Data

In [ ]:
# Load stop points
stops = feather.read_feather(DATA_DIR / 'ALL-StopPoints.feather')
print(f"Loaded {len(stops)} stations")

# Convert coordinates
stops['Easting'] = pd.to_numeric(stops['Place_Location_Easting'])
stops['Northing'] = pd.to_numeric(stops['Place_Location_Northing'])

# Convert to Web Mercator
x_merc, y_merc = osgb_to_webmerc.transform(stops['Easting'].values, stops['Northing'].values)
stops['X_WebMerc'] = x_merc
stops['Y_WebMerc'] = y_merc

# Create coordinate lookup
stop_coords = stops.set_index('AtcoCode')[[
    'Easting', 'Northing', 'X_WebMerc', 'Y_WebMerc', 'Descriptor_CommonName'
]].to_dict('index')

## Load Route Links and Timetable Data

In [ ]:
def parse_duration(duration_str):
    """Parse ISO 8601 duration string (e.g., 'PT1M', 'PT2M30S') to seconds."""
    if pd.isna(duration_str) or duration_str is None:
        return 0
    match = re.match(r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?', str(duration_str))
    if match:
        hours = int(match.group(1) or 0)
        minutes = int(match.group(2) or 0)
        seconds = int(match.group(3) or 0)
        return hours * 3600 + minutes * 60 + seconds
    return 0

def parse_time(time_str):
    """Parse time string to seconds since midnight."""
    if pd.isna(time_str):
        return None
    parts = str(time_str).split(':')
    if len(parts) >= 2:
        hours = int(parts[0])
        minutes = int(parts[1])
        seconds = int(parts[2]) if len(parts) > 2 else 0
        return hours * 3600 + minutes * 60 + seconds
    return None

def load_line_data(line_code):
    """Load all timetable data for a line."""
    prefix = DATA_DIR / f'{line_code}-'
    
    data = {}
    
    # Required files
    files = {
        'journeys': 'VehicleJourneys.feather',
        'patterns': 'JourneyPatterns.feather',
        'timing_links': 'JourneyPatternTimingLinks.feather',
        'route_links': 'RouteLinks.feather',
    }
    
    for key, filename in files.items():
        path = DATA_DIR / f'{line_code}-{filename}'
        if path.exists():
            data[key] = feather.read_feather(path)
        else:
            return None
    
    return data

def get_unique_edges(route_links):
    """Extract unique edges."""
    return route_links[['From_StopPointRef', 'To_StopPointRef']].drop_duplicates()

In [ ]:
# Load route links for all lines
all_edges = {}
all_line_data = {}

print("Loading line data...\n")
for line_code, (color, name) in ALL_LINES.items():
    # Load route links
    route_path = DATA_DIR / f'{line_code}-RouteLinks.feather'
    if route_path.exists():
        links = feather.read_feather(route_path)
        all_edges[line_code] = get_unique_edges(links)
    
    # Load timetable data
    line_data = load_line_data(line_code)
    if line_data:
        all_line_data[line_code] = line_data
        print(f"{name:20s}: {len(line_data['journeys']):5d} journeys")
    else:
        print(f"{name:20s}: No timetable data")

print(f"\nLines with timetables: {len(all_line_data)}")
print(f"Lines with routes: {len(all_edges)}")

## Build Journey Schedules

In [ ]:
def build_journey_schedules(line_data, stop_coords):
    """
    Build train schedules from timetable data.
    Returns list of schedules, each containing (time_seconds, x, y) tuples.
    """
    journeys = line_data['journeys']
    patterns = line_data['patterns']
    timing_links = line_data['timing_links']
    
    # Create lookup from JourneyPattern to JourneyPatternSectionRefs
    if 'JourneyPatternSectionRefs' in patterns.columns:
        pattern_to_sections = patterns.set_index('JourneyPattern')['JourneyPatternSectionRefs'].to_dict()
    else:
        return []
    
    # Group timing links by section
    if 'JourneyPatternSections' in timing_links.columns:
        section_links = timing_links.groupby('JourneyPatternSections')
    else:
        return []
    
    schedules = []
    
    for _, journey in journeys.iterrows():
        jp_ref = journey.get('JourneyPatternRef')
        departure_time = parse_time(journey.get('DepartureTime'))
        
        if departure_time is None or jp_ref not in pattern_to_sections:
            continue
        
        section_ref = pattern_to_sections[jp_ref]
        
        if section_ref not in section_links.groups:
            continue
        
        links = section_links.get_group(section_ref)
        if 'From_SequenceNumber' in links.columns:
            links = links.sort_values('From_SequenceNumber')
        
        schedule = []
        current_time = departure_time
        
        for _, link in links.iterrows():
            from_stop = link.get('From_StopPointRef')
            to_stop = link.get('To_StopPointRef')
            run_time = parse_duration(link.get('RunTime'))
            wait_time = parse_duration(link.get('WaitTime'))
            
            # Add departure from current station
            if from_stop in stop_coords:
                schedule.append({
                    'time': current_time,
                    'x': stop_coords[from_stop]['X_WebMerc'],
                    'y': stop_coords[from_stop]['Y_WebMerc'],
                })
            
            current_time += run_time + wait_time
        
        # Add final station
        if len(links) > 0:
            last_link = links.iloc[-1]
            to_stop = last_link.get('To_StopPointRef')
            if to_stop in stop_coords:
                schedule.append({
                    'time': current_time,
                    'station': to_stop,
                    'x': stop_coords[to_stop]['X_WebMerc'],
                    'y': stop_coords[to_stop]['Y_WebMerc'],
                })
        
        if len(schedule) >= 2:
            schedules.append(schedule)
    
    return schedules

def interpolate_position(schedule, current_time):
    """
    Interpolate train position at given time.
    Returns (x, y) or None if train not running.
    """
    if not schedule:
        return None
    
    first_time = schedule[0]['time']
    last_time = schedule[-1]['time']
    
    if current_time < first_time or current_time > last_time:
        return None
    
    for i in range(len(schedule) - 1):
        t1, t2 = schedule[i]['time'], schedule[i+1]['time']
        if t1 <= current_time <= t2:
            if t2 == t1:
                ratio = 0
            else:
                ratio = (current_time - t1) / (t2 - t1)
            
            x = schedule[i]['x'] + ratio * (schedule[i+1]['x'] - schedule[i]['x'])
            y = schedule[i]['y'] + ratio * (schedule[i+1]['y'] - schedule[i]['y'])
            return (x, y)
    
    return None

In [ ]:
# Build schedules for lines with timetable data
all_schedules = {}

print("Building journey schedules...\n")
for line_code, line_data in all_line_data.items():
    schedules = build_journey_schedules(line_data, stop_coords)
    if schedules:
        all_schedules[line_code] = schedules
        color, name = ALL_LINES[line_code]
        print(f"{name:20s}: {len(schedules):5d} scheduled journeys")

print(f"\nTotal lines with schedules: {len(all_schedules)}")

## Helper Functions for Animation

In [ ]:
def plot_line_webmerc(ax, edges, stop_coords, color, linewidth=2, alpha=0.8):
    """Plot a line using Web Mercator coordinates."""
    for _, edge in edges.iterrows():
        from_stop = edge['From_StopPointRef']
        to_stop = edge['To_StopPointRef']
        
        if from_stop in stop_coords and to_stop in stop_coords:
            x = [stop_coords[from_stop]['X_WebMerc'], stop_coords[to_stop]['X_WebMerc']]
            y = [stop_coords[from_stop]['Y_WebMerc'], stop_coords[to_stop]['Y_WebMerc']]
            ax.plot(x, y, color=color, linewidth=linewidth, alpha=alpha, solid_capstyle='round')

def get_network_stations(edges_dict, stop_coords):
    """Get all stations with valid coordinates."""
    stations = set()
    for edges in edges_dict.values():
        stations.update(edges['From_StopPointRef'].tolist())
        stations.update(edges['To_StopPointRef'].tolist())
    return [s for s in stations if s in stop_coords]

def format_time(seconds):
    """Format seconds since midnight as HH:MM."""
    hours = int(seconds // 3600) % 24
    minutes = int((seconds % 3600) // 60)
    return f"{hours:02d}:{minutes:02d}"

## Create Train Animation

This function creates an animated visualization of trains moving along the network.

In [ ]:
def create_train_animation(
    start_time_str='08:00',
    duration_minutes=60,
    fps=10,
    speed_multiplier=60,
    tile_provider=None,
    title_suffix='',
    filename_suffix='animation',
    lines_to_show=None
):
    """
    Create animation of trains moving through the network.
    
    Args:
        start_time_str: Start time (HH:MM)
        duration_minutes: Real-world duration to animate
        fps: Frames per second
        speed_multiplier: How much faster than real-time (60 = 1 min real = 1 sec animation)
        tile_provider: Map tile provider (default: Esri satellite)
        title_suffix: Additional text for title
        filename_suffix: Suffix for output filename
        lines_to_show: List of line codes to animate (None = all with schedules)
    """
    if tile_provider is None:
        tile_provider = ctx.providers.Esri.WorldImagery
    
    if lines_to_show is None:
        lines_to_show = list(all_schedules.keys())
    
    # Get network bounds
    network_stations = get_network_stations(all_edges, stop_coords)
    all_x = [stop_coords[s]['X_WebMerc'] for s in network_stations]
    all_y = [stop_coords[s]['Y_WebMerc'] for s in network_stations]
    
    margin = 5000
    x_min, x_max = min(all_x) - margin, max(all_x) + margin
    y_min, y_max = min(all_y) - margin, max(all_y) + margin
    
    # Calculate aspect ratio
    x_range = x_max - x_min
    y_range = y_max - y_min
    aspect_ratio = y_range / x_range
    
    base_width = 14
    fig_height = base_width * aspect_ratio
    
    # Time setup
    start_time = parse_time(start_time_str + ':00')
    end_time = start_time + duration_minutes * 60
    
    # Frame calculation
    animation_duration_sec = duration_minutes * 60 / speed_multiplier
    total_frames = int(animation_duration_sec * fps)
    
    print(f"Animation: {duration_minutes} min real-time -> {animation_duration_sec:.1f}s animation")
    print(f"Frames: {total_frames} at {fps} fps")
    print(f"Time range: {format_time(start_time)} - {format_time(end_time)}")
    
    # Set up figure
    fig, ax = plt.subplots(figsize=(base_width, fig_height))
    
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect('equal')
    
    # Draw static background - all lines
    for line_code, edges in all_edges.items():
        color, name = ALL_LINES[line_code]
        alpha = 0.7 if line_code in lines_to_show else 0.3
        lw = 2 if line_code in lines_to_show else 1
        plot_line_webmerc(ax, edges, stop_coords, color, linewidth=lw, alpha=alpha)
    
    # Draw stations (white dots, no landmark identifiers)
    for station in network_stations:
        ax.scatter(stop_coords[station]['X_WebMerc'], stop_coords[station]['Y_WebMerc'],
                  c='white', s=8, edgecolors='#333333', linewidth=0.3, zorder=5)
    
    # Add basemap
    try:
        ctx.add_basemap(ax, source=tile_provider, zoom='auto', alpha=0.7)
    except Exception as e:
        print(f"Basemap error: {e}")
        ax.set_facecolor('#1a1a2e')
    
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Create train scatter plots for each line
    train_scatters = {}
    for line_code in lines_to_show:
        if line_code in all_schedules:
            color, name = ALL_LINES[line_code]
            scatter = ax.scatter([], [], c=color, s=50, edgecolors='white', 
                                linewidth=1, zorder=15, alpha=0.95)
            train_scatters[line_code] = scatter
    
    # Time display
    time_text = ax.text(0.02, 0.98, '', transform=ax.transAxes, fontsize=14,
                        verticalalignment='top', fontweight='bold',
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
                        zorder=20)
    
    # Title
    title = f'London Rail Network{title_suffix}'
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Legend
    legend_elements = [Line2D([0], [0], marker='o', color='w',
                              markerfacecolor=ALL_LINES[code][0],
                              markersize=8, label=ALL_LINES[code][1], linestyle='')
                       for code in lines_to_show if code in train_scatters]
    if legend_elements:
        ax.legend(handles=legend_elements, loc='upper right', fontsize=7, 
                  framealpha=0.9, title='Active Lines')
    
    def init():
        for scatter in train_scatters.values():
            scatter.set_offsets(np.empty((0, 2)))
        time_text.set_text('')
        return list(train_scatters.values()) + [time_text]
    
    def animate(frame):
        progress = frame / total_frames
        current_time = start_time + progress * (end_time - start_time)
        
        # Update time display
        time_text.set_text(format_time(current_time))
        
        # Update train positions
        for line_code, scatter in train_scatters.items():
            if line_code not in all_schedules:
                continue
            
            positions = []
            for schedule in all_schedules[line_code]:
                pos = interpolate_position(schedule, current_time)
                if pos:
                    positions.append(pos)
            
            if positions:
                scatter.set_offsets(positions)
            else:
                scatter.set_offsets(np.empty((0, 2)))
        
        return list(train_scatters.values()) + [time_text]
    
    print("\nGenerating animation...")
    anim = animation.FuncAnimation(fig, animate, init_func=init,
                                   frames=total_frames, interval=1000/fps, blit=True)
    
    # Save as GIF
    output_path = OUTPUT_DIR / f'train_{filename_suffix}.gif'
    print(f"Saving to {output_path}...")
    anim.save(str(output_path), writer='pillow', fps=fps)
    plt.close()
    
    print(f"Done! Saved to {output_path}")
    return output_path

## Monday Morning Rush Hour

In [ ]:
# Monday morning rush hour animation
monday_path = create_train_animation(
    start_time_str='08:00',
    duration_minutes=30,
    fps=10,
    speed_multiplier=60,
    title_suffix='\nMonday Morning Rush Hour (08:00-08:30)',
    filename_suffix='monday_morning'
)

# Display the animation
from IPython.display import Image
Image(filename=str(monday_path))

## Friday Evening Rush Hour

In [ ]:
# Friday evening rush hour
friday_path = create_train_animation(
    start_time_str='17:30',
    duration_minutes=30,
    fps=10,
    speed_multiplier=60,
    title_suffix='\nFriday Evening Rush Hour (17:30-18:00)',
    filename_suffix='friday_evening'
)

Image(filename=str(friday_path))

## Saturday Midday

In [ ]:
# Saturday midday
saturday_path = create_train_animation(
    start_time_str='12:00',
    duration_minutes=30,
    fps=10,
    speed_multiplier=60,
    title_suffix='\nSaturday Midday (12:00-12:30)',
    filename_suffix='saturday_midday'
)

Image(filename=str(saturday_path))

## Sunday Morning

In [ ]:
# Sunday morning (quieter service)
sunday_path = create_train_animation(
    start_time_str='10:00',
    duration_minutes=30,
    fps=10,
    speed_multiplier=60,
    title_suffix='\nSunday Morning (10:00-10:30)',
    filename_suffix='sunday_morning'
)

Image(filename=str(sunday_path))

## Central London Zoom - Rush Hour

In [ ]:
def create_central_london_animation(
    start_time_str='08:00',
    duration_minutes=30,
    fps=10,
    speed_multiplier=60,
    filename_suffix='central'
):
    """Create animation zoomed on central London."""
    
    # Central London bounds
    center_lon, center_lat = -0.12, 51.51
    center_x, center_y = wgs84_to_webmerc.transform(center_lon, center_lat)
    
    radius = 6000
    x_min, x_max = center_x - radius, center_x + radius
    y_min, y_max = center_y - radius, center_y + radius
    
    # Square figure for central view
    fig, ax = plt.subplots(figsize=(14, 14))
    
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect('equal')
    
    # Draw lines
    for line_code, edges in all_edges.items():
        color, name = ALL_LINES[line_code]
        plot_line_webmerc(ax, edges, stop_coords, color, linewidth=3, alpha=0.8)
    
    # Draw stations
    network_stations = get_network_stations(all_edges, stop_coords)
    for station in network_stations:
        x = stop_coords[station]['X_WebMerc']
        y = stop_coords[station]['Y_WebMerc']
        if x_min <= x <= x_max and y_min <= y <= y_max:
            ax.scatter(x, y, c='white', s=20, edgecolors='#333333', linewidth=0.5, zorder=5)
    
    # Add basemap
    try:
        ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, zoom=14, alpha=0.75)
    except Exception as e:
        print(f"Basemap error: {e}")
    
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Time setup
    start_time = parse_time(start_time_str + ':00')
    end_time = start_time + duration_minutes * 60
    animation_duration_sec = duration_minutes * 60 / speed_multiplier
    total_frames = int(animation_duration_sec * fps)
    
    # Train scatters
    train_scatters = {}
    for line_code in all_schedules.keys():
        color, name = ALL_LINES[line_code]
        scatter = ax.scatter([], [], c=color, s=80, edgecolors='white', 
                            linewidth=1.5, zorder=15, alpha=0.95)
        train_scatters[line_code] = scatter
    
    # Time display
    time_text = ax.text(0.02, 0.98, '', transform=ax.transAxes, fontsize=16,
                        verticalalignment='top', fontweight='bold',
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9),
                        zorder=20)
    
    ax.set_title('Central London Rail Network', fontsize=14, fontweight='bold')
    
    def init():
        for scatter in train_scatters.values():
            scatter.set_offsets(np.empty((0, 2)))
        time_text.set_text('')
        return list(train_scatters.values()) + [time_text]
    
    def animate(frame):
        progress = frame / total_frames
        current_time = start_time + progress * (end_time - start_time)
        time_text.set_text(format_time(current_time))
        
        for line_code, scatter in train_scatters.items():
            positions = []
            for schedule in all_schedules.get(line_code, []):
                pos = interpolate_position(schedule, current_time)
                if pos:
                    x, y = pos
                    # Only show trains in view
                    if x_min <= x <= x_max and y_min <= y <= y_max:
                        positions.append(pos)
            
            if positions:
                scatter.set_offsets(positions)
            else:
                scatter.set_offsets(np.empty((0, 2)))
        
        return list(train_scatters.values()) + [time_text]
    
    print(f"Creating central London animation...")
    anim = animation.FuncAnimation(fig, animate, init_func=init,
                                   frames=total_frames, interval=1000/fps, blit=True)
    
    output_path = OUTPUT_DIR / f'train_central_{filename_suffix}.gif'
    anim.save(str(output_path), writer='pillow', fps=fps)
    plt.close()
    
    print(f"Saved to {output_path}")
    return output_path

In [ ]:
# Central London rush hour
central_path = create_central_london_animation(
    start_time_str='08:00',
    duration_minutes=30,
    fps=10,
    speed_multiplier=60,
    filename_suffix='rush_hour'
)

Image(filename=str(central_path))

## Full Day Time-Lapse (Condensed)

In [ ]:
# Full day timelapse (06:00 - 00:00, 18 hours in 30 seconds)
fullday_path = create_train_animation(
    start_time_str='06:00',
    duration_minutes=18 * 60,  # 18 hours
    fps=15,
    speed_multiplier=18 * 60 * 2,  # 18 hours in 30 seconds
    title_suffix='\nFull Day Time-Lapse (06:00-00:00)',
    filename_suffix='fullday_timelapse'
)

Image(filename=str(fullday_path))

## Summary

In [ ]:
print("=" * 60)
print("TRAIN ANIMATION SUMMARY")
print("=" * 60)
print(f"\nLines with timetable data: {len(all_schedules)}")
print(f"Total scheduled journeys: {sum(len(s) for s in all_schedules.values())}")
print(f"\nAnimations created:")
print(f"  - Monday morning rush hour")
print(f"  - Friday evening rush hour")
print(f"  - Saturday midday")
print(f"  - Sunday morning")
print(f"  - Central London rush hour")
print(f"  - Full day time-lapse")
print(f"\nOutput directory: {OUTPUT_DIR}")
print("=" * 60)